# XGBoost

In [ ]:
%pip install -q mlflow xgboost scikit-learn pandas matplotlib

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    pass
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# თუ GPU არ გაქვთ, False დააყენეთ
USE_GPU = True
DEVICE = "cuda" if USE_GPU else "cpu"
print("device:", DEVICE)

from src.data import load_raw, submission_ids
from src.features import WalmartFeatureBuilder, MARKDOWN_COLS
from src.metrics import wmae, holiday_weights
from src.validation import year_back_split, time_folds, DEV_TRAIN_END, TRAIN_END
from src.tracking import setup_mlflow, REGISTERED_MODEL_NAME

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("MLflow tracking:", setup_mlflow("XGBoost_Training"))

In [ ]:
raw = load_raw()
train, test = raw["train"], raw["test"]

tr_idx, va_idx = year_back_split(train["Date"])
dev_train, dev_valid = train.iloc[tr_idx], train.iloc[va_idx]
print(f"train: {train.shape}, test: {test.shape}")
print(f"dev_train: {len(dev_train):,} რიგი (<= {DEV_TRAIN_END.date()}), "
      f"dev_valid: {len(dev_valid):,} რიგი (39 კვირა, შეიცავს Thanksgiving/Christmas/Super Bowl-ს)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

weekly = train.groupby("Date").agg(total=("Weekly_Sales", "sum"),
                                   holiday=("IsHoliday", "max"))
axes[0].plot(weekly.index, weekly["total"] / 1e6, lw=1.2)
for d in weekly.index[weekly["holiday"] == 1]:
    axes[0].axvline(d, color="tomato", alpha=0.35, lw=1)
axes[0].set_title("Total weekly sales, $M (red = 5x holiday weeks)")

series_mean = train.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
axes[1].hist(np.log10(series_mean.clip(lower=1)), bins=50, color="#3b6ea5")
axes[1].set_title("log10(mean weekly sales) per (store, dept)")
axes[1].set_xlabel("orders of magnitude apart")

md_share = (raw["features"].set_index("Date")[MARKDOWN_COLS].notna().any(axis=1)
            .groupby(level=0).mean())
axes[2].plot(pd.to_datetime(md_share.index), md_share.values, color="#c77d2e")
axes[2].axvline(pd.Timestamp("2012-10-26"), color="gray", ls="--", lw=1)
axes[2].set_title("Share of stores with MarkDown data (--- = train end)")
plt.tight_layout(); plt.show()

print(f"უარყოფითი გაყიდვები (დაბრუნებები): {(train['Weekly_Sales'] < 0).sum():,} რიგი")
print(f"სადღესასწაულო კვირების წილი: {train['IsHoliday'].mean():.1%}")

In [ ]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    mlflow.log_params({
        "negative_sales_policy": "keep (real returns)",
        "markdown_nan_policy": "fill 0 + has_markdown flag",
        "cpi_unemployment_policy": "per-store ffill/bfill",
        "missing_weeks_policy": "no row (tabular model)",
    })
    mlflow.log_metrics({
        "n_negative_sales": int((train["Weekly_Sales"] < 0).sum()),
        "markdown_row_coverage": float(raw["features"][MARKDOWN_COLS].notna().any(axis=1).mean()),
        "n_series": int(train.groupby(["Store", "Dept"]).ngroups),
    })

In [ ]:
BASE_PARAMS = dict(n_estimators=800, learning_rate=0.08, max_depth=8,
                   subsample=0.9, colsample_bytree=0.8,
                   objective="reg:absoluteerror",
                   tree_method="hist", device=DEVICE,
                   random_state=RANDOM_STATE, n_jobs=-1)

def make_pipe(**extra_params):
    params = dict(BASE_PARAMS)
    params.update(extra_params)
    return Pipeline([
        ("features", WalmartFeatureBuilder(features_df=raw["features"],
                                           stores_df=raw["stores"])),
        ("model", XGBRegressor(**params)),
    ])

def eval_year_back(pipe):

    pipe.fit(dev_train, dev_train["Weekly_Sales"],
             model__sample_weight=holiday_weights(dev_train["IsHoliday"]))
    pred = pipe.predict(dev_valid.drop(columns=["Weekly_Sales"]))
    return wmae(dev_valid["Weekly_Sales"], pred, dev_valid["IsHoliday"])

with mlflow.start_run(run_name="XGBoost_Baseline"):
    base_pipe = make_pipe()
    base_wmae = eval_year_back(base_pipe)
    mlflow.log_metric("valid_wmae", base_wmae)
print(f"Baseline valid WMAE: {base_wmae:.0f}")

In [ ]:
with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    fb = base_pipe.named_steps["features"]
    names = fb.get_feature_names_out()
    imp = pd.Series(base_pipe.named_steps["model"].feature_importances_,
                    index=names).sort_values()
    fig, ax = plt.subplots(figsize=(7, 9))
    imp.plot.barh(ax=ax)
    ax.set_title("XGBoost feature importance")
    plt.tight_layout()
    mlflow.log_figure(fig, "feature_importance.png")
    plt.show()

    weak = list(imp.index[:8])
    print("ვცდით ამ feature ების გარეშე:", weak)
    X_tr = fb.transform(dev_train).drop(columns=weak)
    X_va = fb.transform(dev_valid).drop(columns=weak)
    model_sel = XGBRegressor(**BASE_PARAMS)
    model_sel.fit(X_tr, dev_train["Weekly_Sales"],
                  sample_weight=holiday_weights(dev_train["IsHoliday"]))
    pred = model_sel.predict(X_va)
    sel_wmae = wmae(dev_valid["Weekly_Sales"], pred, dev_valid["IsHoliday"])

    mlflow.log_param("dropped", ", ".join(weak))
    mlflow.log_metrics({"valid_wmae_full": base_wmae, "valid_wmae_selected": sel_wmae})

print(f"ყველა feature ით: {base_wmae:.0f} | სუსტების გარეშე: {sel_wmae:.0f}")

In [ ]:
param_grid = [
    dict(max_depth=6,  learning_rate=0.10, n_estimators=800),
    dict(max_depth=8,  learning_rate=0.08, n_estimators=800),
    dict(max_depth=8,  learning_rate=0.05, n_estimators=1500),
    dict(max_depth=10, learning_rate=0.05, n_estimators=1500, min_child_weight=10),
]

best_cv, best_params = None, None
with mlflow.start_run(run_name="XGBoost_CV"):
    for i, p in enumerate(param_grid):
        with mlflow.start_run(run_name=f"config_{i}", nested=True):
            fold_scores = []
            for tri, vai in time_folds(train["Date"], n_folds=3, valid_weeks=13):
                ftr, fva = train.iloc[tri], train.iloc[vai]
                pipe = make_pipe(**p)
                pipe.fit(ftr, ftr["Weekly_Sales"],
                         model__sample_weight=holiday_weights(ftr["IsHoliday"]))
                pred = pipe.predict(fva.drop(columns=["Weekly_Sales"]))
                fold_scores.append(wmae(fva["Weekly_Sales"], pred, fva["IsHoliday"]))
            cv_mean = float(np.mean(fold_scores))
            mlflow.log_params(p)
            mlflow.log_metric("cv_wmae_mean", cv_mean)
            print(f"config_{i}: {p} -> CV WMAE {cv_mean:.0f}")
            if best_cv is None or cv_mean < best_cv:
                best_cv, best_params = cv_mean, p

print(f"\nსაუკეთესო: {best_params} (CV WMAE {best_cv:.0f})")

In [ ]:
with mlflow.start_run(run_name="XGBoost_Final") as run:
    final_pipe = make_pipe(**best_params)
    final_wmae = eval_year_back(final_pipe)
    mlflow.log_params(best_params)
    mlflow.log_metric("valid_wmae", final_wmae)

    final_pipe.fit(train, train["Weekly_Sales"],
                   model__sample_weight=holiday_weights(train["IsHoliday"]))

    mlflow.sklearn.log_model(final_pipe, name="model", code_paths=["src"],
                             serialization_format="cloudpickle")
    xgb_final_run_id = run.info.run_id

print(f"საბოლოო valid WMAE: {final_wmae:.0f} | run: {xgb_final_run_id}")

In [ ]:
REGISTER_AS_CHAMPION = False

if REGISTER_AS_CHAMPION:
    from mlflow import MlflowClient
    mv = mlflow.register_model(f"runs:/{xgb_final_run_id}/model", REGISTERED_MODEL_NAME)
    MlflowClient().set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", mv.version)
    print(f"დარეგისტრირდა: {REGISTERED_MODEL_NAME} v{mv.version} (alias: champion)")